## Imports & Initialization:

In [60]:
%load_ext autoreload
%autoreload 2

import os
import sys
import random
from dotenv.main import load_dotenv

load_dotenv()
REPO_PATH = os.getenv('REPO_PATH')

sys.path.append(REPO_PATH)

from knowledge_maze.tools.utils import load_json, display_usage_log
from knowledge_maze.evaluation.models.choose_models import KnowledgeMazeRunner
from knowledge_maze.evaluation.utils.preprocess import KnowledgeMazeDatasetDistiller
from knowledge_maze.evaluation.models.model_inference import KnowledgeMazeIncrementalRunner

question_data = load_json(os.path.join(REPO_PATH, "data_files/knowledge_maze_questions.json"))
hedged_question_data = load_json(os.path.join(REPO_PATH, "data_files/knowledge_maze_questions_hedged.json"))
paragraphs = load_json(os.path.join(REPO_PATH, "data_files/knowledge_maze_paragraphs.json"))

CONFIG_PATH = os.path.join(REPO_PATH, "knowledge_maze/evaluation/models/models_config.yaml")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Preliminary Reliability Assessment

Before conducting the full-scale evaluation of **CLECK**, we perform a preliminary reliability check on each model to ensure it possesses the baseline reasoning capabilities required for the benchmark.

### Methodology

* **Stratified Sampling:** We evaluate each model on a subset of **550 random samples**, stratified across our eleven reasoning operations to prevent logical type bias.
* **Reasoning Verification:** To ensure the model can handle the underlying logic of the task, we execute **three Self-Consistency (SC) rounds** for each reasoning paragraph in isolation.
* **Success Criteria:** A model is considered reliable if it achieves an **overall success rate above 90%** across all test cases.
* **General Success Metric:** Additionally, the model must maintain a **general success rate above 90%**, defined as succeeding in at least **2 out of 3 SC rounds** for any given sample.
* **Validation Goal:** This process confirms that the model’s later failures are due to the **latent conflict** rather than a fundamental inability to perform the multi-hop reasoning or the specific mathematical operations, such as counting or filtering.

---

### The "Diagnostic" Logic

By validating reasoning on isolated hops first, we establish a **capability baseline**. This allows us to argue that when a model defaults to an authoritative claim in the full benchmark, it is a behavioral "lock" rather than a lack of logical skill.

In [11]:
runner = KnowledgeMazeRunner(limit=550)
results_df = await runner.execute(question_data, paragraphs)

✅ Stratified 550 samples across 11 strategies.
🚀 Starting CONSISTENCY sweep: 7 models, 550 samples.

🔍 Evaluating Consistency: google/gemini-3-flash-preview...


Inferencing google/gemini-3-flash-preview:   0%|          | 0/3300 [00:00<?, ?it/s]

⚖️ Sending 2891 deterministic failures to Judge for global review...


Judge Batch Review:   0%|          | 0/2891 [00:00<?, ?it/s]

API Error: 1 validation error for CoTResponse
  Invalid JSON: expected value at line 1 column 1 [type=json_invalid, input_value='H{\n  "explanation": "Th...\n  "answer": "True"\n}', input_type=str]
    For further information visit https://errors.pydantic.dev/2.10/v/json_invalid
API Error: 1 validation error for CoTResponse
  Invalid JSON: expected value at line 1 column 1 [type=json_invalid, input_value='H{\n  "explanation": "Th...\n  "answer": "True"\n}', input_type=str]
    For further information visit https://errors.pydantic.dev/2.10/v/json_invalid
API Error: 1 validation error for CoTResponse
  Invalid JSON: expected value at line 1 column 1 [type=json_invalid, input_value='H{\n  "explanation": "Th...\n  "answer": "True"\n}', input_type=str]
    For further information visit https://errors.pydantic.dev/2.10/v/json_invalid
API Error: 1 validation error for CoTResponse
  Invalid JSON: EOF while parsing a string at line 2 column 27 [type=json_invalid, input_value='{\n  "explanation"

...

Inferencing deepseek/deepseek-v4-pro:   0%|          | 0/3300 [00:00<?, ?it/s]

⚖️ Sending 1593 deterministic failures to Judge for global review...


Judge Batch Review:   0%|          | 0/1593 [00:00<?, ?it/s]

API Error: 1 validation error for CoTResponse
  Invalid JSON: EOF while parsing a string at line 2 column 28 [type=json_invalid, input_value='{\n  "explanation": "The ground', input_type=str]
    For further information visit https://errors.pydantic.dev/2.10/v/json_invalid
API Error: 1 validation error for CoTResponse
  Invalid JSON: EOF while parsing a string at line 2 column 3 [type=json_invalid, input_value='{\n  "', input_type=str]
    For further information visit https://errors.pydantic.dev/2.10/v/json_invalid
API Error: 1 validation error for CoTResponse
  Invalid JSON: EOF while parsing a string at line 2 column 225 [type=json_invalid, input_value='{\n  "explanation": "The...ctually incorrect based', input_type=str]
    For further information visit https://errors.pydantic.dev/2.10/v/json_invalid
API Error: 1 validation error for CoTResponse
  Invalid JSON: EOF while parsing a string at line 2 column 21 [type=json_invalid, input_value='{\n  "explanation": "The', input_type=str]

Model Architecture,Mean Success,General (≥2/3),Strict (3/3),Stability Profile (3 to 0)
qwen/qwen3.5-flash-02-23,98.0%,98.2%,96.9%,533 7 4 6
openai/gpt-5-mini,96.7%,97.1%,94.4%,519 15 8 8
deepseek/deepseek-v4-pro,95.2%,97.6%,88.2%,485 52 11 2
meta-llama/llama-4-maverick,94.1%,95.1%,90.4%,497 26 10 17
google/gemini-3-flash-preview,94.1%,94.7%,91.3%,502 19 8 21
openai/gpt-5.4,93.3%,93.5%,91.1%,501 13 10 26
mistralai/mistral-medium-3.1,85.9%,86.5%,81.5%,448 28 18 56


In [53]:
runner_2 = KnowledgeMazeRunner(limit=550)
results_df_2 = await runner.execute(question_data, paragraphs)

✅ Stratified 550 samples across 11 strategies.
🚀 Starting CONSISTENCY sweep: 1 models, 550 samples.

🔍 Evaluating Consistency: deepseek/deepseek-v4-flash...


Inferencing deepseek/deepseek-v4-flash:   0%|          | 0/3300 [00:00<?, ?it/s]

⚖️ Sending 689 deterministic failures to Judge for global review...


Judge Batch Review:   0%|          | 0/689 [00:00<?, ?it/s]

Model Architecture,Mean Success,General (≥2/3),Strict (3/3),Stability Profile (3 to 0)
deepseek/deepseek-v4-flash,94.1%,96.4%,87.6%,482 48 11 9


In [62]:
runner_3 = KnowledgeMazeRunner(limit=550)
results_df_3 = await runner_3.execute(question_data, paragraphs)

✅ Stratified 550 samples across 11 strategies.
🚀 Starting CONSISTENCY sweep: 3 models, 550 samples.

🔍 Evaluating Consistency: z-ai/glm-4.7-flash...


Inferencing z-ai/glm-4.7-flash:   0%|          | 0/3300 [00:00<?, ?it/s]

API Error: Expecting value: line 149 column 1 (char 814)
API Error: Expecting value: line 167 column 1 (char 913)
⚖️ Sending 1460 deterministic failures to Judge for global review...


Judge Batch Review:   0%|          | 0/1460 [00:00<?, ?it/s]

🔍 Evaluating Consistency: openai/gpt-5.1...


...

Inferencing deepseek/deepseek-r1-0528:   0%|          | 0/3300 [00:00<?, ?it/s]

⚖️ Sending 3299 deterministic failures to Judge for global review...


Judge Batch Review:   0%|          | 0/3299 [00:00<?, ?it/s]

API Error: 1 validation error for CoTResponse
  Invalid JSON: expected value at line 1 column 1 [type=json_invalid, input_value='H{\n  "explanation": "Th...\n  "answer": "True"\n}', input_type=str]
    For further information visit https://errors.pydantic.dev/2.10/v/json_invalid


Model Architecture,Mean Success,General (≥2/3),Strict (3/3),Stability Profile (3 to 0)
deepseek/deepseek-r1-0528,97.2%,98.7%,93.3%,513 30 4 3
z-ai/glm-4.7-flash,93.2%,94.9%,87.1%,479 43 15 13
openai/gpt-5.1,92.1%,92.7%,87.5%,481 29 18 22


In [69]:
import pandas as pd
final_df = pd.concat([results_df_3, results_df_2, results_df])
runner.sweep.display_dashboard(final_df)

Model Architecture,Mean Success,General (≥2/3),Strict (3/3),Stability Profile (3 to 0)
deepseek/deepseek-r1-0528,97.2%,98.7%,93.3%,513 30 4 3
z-ai/glm-4.7-flash,93.2%,94.9%,87.1%,479 43 15 13
openai/gpt-5.1,92.1%,92.7%,87.5%,481 29 18 22
deepseek/deepseek-v4-flash,94.1%,96.4%,87.6%,482 48 11 9
qwen/qwen3.5-flash-02-23,98.0%,98.2%,96.9%,533 7 4 6
openai/gpt-5-mini,96.7%,97.1%,94.4%,519 15 8 8
deepseek/deepseek-v4-pro,95.2%,97.6%,88.2%,485 52 11 2
meta-llama/llama-4-maverick,94.1%,95.1%,90.4%,497 26 10 17
google/gemini-3-flash-preview,94.1%,94.7%,91.3%,502 19 8 21
openai/gpt-5.4,93.3%,93.5%,91.1%,501 13 10 26


# 🛠️ Preprocessing Dataset:

### 1. Stratified Population Sampling

To ensure a balanced benchmark, the distiller samples the dataset based on specific research constraints:

* **Hedged Priority**: Automatically includes all **803 Hedged samples** to ensure the uncertainty study is fully represented.
* **Strategy Balancing**: Targets **300 samples per reasoning strategy** (e.g., Temporal Ordering, Multi-hop) for the Full Study.
* **Strict Subsetting**: The **CoT (1.1k)** and **Hedged (803)** datasets are generated as strict subsets of the **Full Study (3.2k)**, allowing for direct paired-sample comparisons.

### 2. Deterministic Context Shuffling

Instead of shuffling paragraphs during the live experiment, the distiller pre-calculates and saves **5 frozen iterations** for every sample:

* **Context IDs**: Each iteration stores a specific list of 10 IDs (1 Evidence, 1 Claim, 8 Distractors) in a fixed sequence.
* **Consistency**: Every model (Gemini, GPT-5, etc.) will see the exact same 5 layouts for Question $X$, ensuring that any performance difference is due to the model architecture, not a "lucky" or "unlucky" shuffle.

### 3. Positional Bias Guardrails

The process includes an automated audit to verify that the shuffling is mathematically "fair":

* **Primacy vs. Recency**: Checks that the Claim does not appear before the Evidence more often than chance (aiming for a ~50/50 split).
* **Edge Salience**: Ensures that both the Truth (Evidence) and the Lie (Claim) occupy high-attention "edge" positions (start/end of context) with equal frequency.

In [14]:
distiller = KnowledgeMazeDatasetDistiller(question_data, paragraphs)
distiller.create_subsets()


════════════════════════════════════════
🚀 PREPROCESSING DASHBOARD SUMMARY
════════════════════════════════════════
Dataset Name         | Total Samples   |
----------------------------------------
full_study.json      | 3132            |
cot_subset.json      | 1100            |
hedged_subset.json   | 801             |
════════════════════════════════════════

### Strategy Distribution Report

| Selected Strategy              | Full Study         | CoT Subset         | Hedged Subset      |
| ------------------------------ | ------------------ | ------------------ | ------------------ |
| AGGREGATION                    | 169 (5.4%)         | 100 (9.1%)         | 16 (2.0%)          |
| CATEGORICAL_FILTER_LOOKUP      | 300 (9.6%)         | 100 (9.1%)         | 63 (7.9%)          |
| COMPARATIVE_DIFFERENCE         | 300 (9.6%)         | 100 (9.1%)         | 47 (5.9%)          |
| COUNT_TOTAL                    | 300 (9.6%)         | 100 (9.1%)         | 227 (28.3%)        |
| EXTREME_NUME

# Run qa

In [17]:
base_question_dataset = load_json(os.path.join(REPO_PATH, "results/preprocessed_data/full_study.json"))
cot_question_dataset = load_json(os.path.join(REPO_PATH, "results/preprocessed_data/cot_subset.json"))
hedge_question_dataset = load_json(os.path.join(REPO_PATH, "results/preprocessed_data/hedged_subset.json"))

In [52]:
base_runner = KnowledgeMazeIncrementalRunner(
    config_path=CONFIG_PATH,
    dataset_dict=base_question_dataset,
    paragraphs_data=paragraphs,
    prompt_type="baseline"
)
await base_runner.run_incremental()


>>> Model: deepseek/deepseek-v4-flash | Mode: STANDARD
--- Found 3132 pending samples. Generating frozen prompts...


Inference: deepseek/deepseek-v4-flash:   0%|          | 0/15660 [00:00<?, ?it/s]

CancelledError: 

In [ ]:
cot_runner = KnowledgeMazeIncrementalRunner(
    config_path=CONFIG_PATH,
    dataset_dict=cot_question_dataset,
    paragraphs_data=paragraphs,
    prompt_type="cot"
)
await cot_runner.run_incremental(sample_limit=10)

In [ ]:
hedge_runner = KnowledgeMazeIncrementalRunner(
    config_path=CONFIG_PATH,
    dataset_dict=hedge_question_dataset,
    paragraphs_data=paragraphs,
    prompt_type="baseline",
    use_hedged_text=True
)
await hedge_runner.run_incremental(sample_limit=10)

# Asses Ans with Judge